# Verifica concettuale di Lamezia Trasparente
Issue #1115, 7 settembre 2026. Lo snapshot allegato proviene da query in sola lettura; questo notebook ne ricalcola gli esiti senza accedere alla produzione. Non sommare granularità diverse e non interpretare i conteggi come completezza.

Le query SQL originali sono conservate accanto al notebook. La prima conta tutte le 70 tabelle applicative registrate; la seconda verifica CUP vuoti o ripetuti in Attuazione PNRR. Le due letture non costituiscono uno snapshot transazionale unico.

In [1]:
import json
from pathlib import Path

candidates = [Path.cwd(), Path.cwd() / 'docs/architecture/audit']
audit_dir = next(p for p in candidates if (p / 'conceptual-2026-09-07.json').exists())
evidence = json.loads((audit_dir / 'conceptual-2026-09-07.json').read_text())
counts = {row['table_name']: row['rows'] for row in evidence['counts']}
assert len(counts) == len(evidence['counts']), 'Duplicate table counts'
print({'tables': len(counts), 'populated': sum(n > 0 for n in counts.values()), 'empty': sum(n == 0 for n in counts.values())})
print(evidence['pnrrOverlap'][0])

{'tables': 70, 'populated': 12, 'empty': 58}
{'distinct_nonempty_cup': 30, 'repeated_cup_groups': 0, 'source_rows': 30, 'without_cup': 0}


In [2]:
for table, rows in counts.items():
    if rows:
        print(f'{table}: {rows}')

for table in ['canonical_subjects', 'legacy_subject_map', 'publications', 'contracts', 'crime_public_events']:
    print(f'{table}: {counts[table]} (non certifica la copertura dei contenuti del sito)')

attuazione_pnrr_projects: 30
crime_event_locations: 33
crime_event_offences: 40
crime_event_sources: 342
crime_events: 24
crime_sources: 86
source_acquisition_runs: 45
source_artifacts: 5
source_endpoints: 5
source_records: 260
source_releases: 5
source_sources: 5
canonical_subjects: 0 (non certifica la copertura dei contenuti del sito)
legacy_subject_map: 0 (non certifica la copertura dei contenuti del sito)
publications: 0 (non certifica la copertura dei contenuti del sito)
contracts: 0 (non certifica la copertura dei contenuti del sito)
crime_public_events: 0 (non certifica la copertura dei contenuti del sito)


## Ripetizione della verifica
Le seguenti stringhe sono le query effettivamente usate. La funzione opzionale accetta una connessione psycopg già configurata in modo sicuro, su cui sia possibile iniziare una nuova transazione; non crea connessioni, non legge né stampa credenziali e non viene chiamata automaticamente. Per una nuova rilevazione conservare esiti e data in un nuovo snapshot, senza sostituire quello storico.

In [3]:
counts_sql = (audit_dir / 'conceptual-counts-2026-09-07.sql').read_text()
pnrr_sql = (audit_dir / 'conceptual-pnrr-2026-09-07.sql').read_text()

def repeat_readonly(connection):
    with connection.transaction():
        with connection.cursor() as cursor:
            cursor.execute('SET TRANSACTION ISOLATION LEVEL REPEATABLE READ, READ ONLY')
            cursor.execute("SET LOCAL statement_timeout = '30s'")
            result = {}
            for name, query in [('counts', counts_sql), ('pnrrOverlap', pnrr_sql)]:
                cursor.execute(query)
                columns = [column.name for column in cursor.description]
                result[name] = [dict(zip(columns, row)) for row in cursor.fetchall()]
            return result

print('Query conservate; nessuna connessione o modifica al database eseguita dal notebook.')

Query conservate; nessuna connessione o modifica al database eseguita dal notebook.


## Decisione
Le rappresentazioni PNRR sono distinte per fonte e non vanno cancellate perché hanno lo stesso tipo concettuale. Il progetto canonico, le asserzioni di fonte e la risoluzione sono lavoro residuo. La revisione introduce la mappa concettuale e i controlli di copertura; non dichiara conclusa la migrazione dei contenuti. Vedere `../conceptual-assessment-2026-09-07.md` e il catalogo generato dal registro unico.